# 🤖 Week 1 — LLM Fundamentals, Real Behaviours & First Chatbot
### A beginner's guide with live experiments

---

**By the end of this notebook you will understand:**
- What a token is, why it matters, and how to estimate real costs
- How an LLM predicts the next word (and why it sometimes gets it wrong)
- What system prompt and user prompt are — and how to control output format
- How to talk to GPT-4o-mini AND Claude Haiku in the same notebook
- Why models give confident wrong answers (hallucination) — 3 patterns with ground truth
- How to automatically detect unreliable answers with self-consistency
- How to fix stale answers using Tavily live search
- How to build a persistent multi-turn chatbot

**No prior AI experience needed. Every cell is explained before you run it.**

---

### 📦 Libraries used
| Library | What it does |
|---|---|
| `openai` | Talks to GPT models |
| `anthropic` | Talks to Claude models |
| `tiktoken` | Counts tokens (OpenAI's tokeniser) |
| `tavily-python` | Live web search — gives models fresh facts |
| `python-dotenv` | Loads API keys from a `.env` file |
| `requests` | HTTP calls (Wikipedia ground truth) |

---
### Agenda
1. Setup — install, API keys, shared utilities
2. Tokens & cost — what your budget actually buys
3. Temperature — the determinism dial
4. System prompt vs User prompt — shaping the model
5. Two models, same question — GPT vs Claude
6. Hallucination anatomy — 3 patterns with ground truth verification
7. Self-consistency — detecting unreliable answers automatically
8. Fix stale answers with Tavily live search
9. Interactive prompt lab — exercises
10. Chatbot — persistent multi-turn conversation with history


---
## ⚙️ Cell 0 — Install & Setup
Run this first. It installs everything and loads your API keys.

In [11]:
# Install all libraries we need (pydantic required by shared/utils.py)
!pip install openai anthropic tiktoken tavily-python python-dotenv requests pydantic --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vivik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
import sys, os, json, math, textwrap, requests
from dotenv import load_dotenv
from IPython.display import display, Markdown
import tiktoken

# Load keys from .env file in the same folder
# Your .env file should look like this:
#   OPENAI_API_KEY=sk-...
#   ANTHROPIC_API_KEY=sk-ant-...
#   TAVILY_API_KEY=tvly-...
load_dotenv()

OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY",    "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
TAVILY_API_KEY    = os.getenv("TAVILY_API_KEY",    "")

print("API Key Status:")
print(f"  OpenAI   : {'✅ loaded' if OPENAI_API_KEY    else '❌ missing — add to .env'}")
print(f"  Anthropic: {'✅ loaded' if ANTHROPIC_API_KEY else '❌ missing — add to .env'}")
print(f"  Tavily   : {'✅ loaded' if TAVILY_API_KEY    else '❌ missing — get free key at tavily.com'}")

from openai import OpenAI
import anthropic

oai_client    = OpenAI(api_key=OPENAI_API_KEY)
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print()
print("✅  Raw clients ready!")


API Key Status:
  OpenAI   : ✅ loaded
  Anthropic: ✅ loaded
  Tavily   : ✅ loaded

✅  Raw clients ready!


### 🎓 Trainer note — mixed API style (on purpose)

**Week 1** keeps most calls on the **raw OpenAI / Anthropic SDKs** so you see exactly what each provider expects (`messages`, `system`, `temperature`, etc.).

**Shared utils** (`LLMClient`, `Guardrails`, `Tracer`) appear in a **few cells only** — a preview of what Week 2 standardises on.

| Style | Where | Why |
|---|---|---|
| Raw SDK | Sections 1–5, 7–9 | Learn the plumbing |
| `LLMClient` / `Guardrails` | Cost calc, self-consistency, session summary | See tracing & production helpers |

The session tracer at the end will **not** list every call — only those routed through `LLMClient`. That is expected for Week 1.


In [13]:
# ── Load shared utilities (selective use in Week 1) ─────────────────────────
# Week 1 mostly uses raw oai_client / claude_client above.
# LLMClient + Guardrails appear in a few cells; Week 2 uses them throughout.

def _find_repo_root(*start_dirs):
    for start in start_dirs:
        if not start:
            continue
        root = os.path.abspath(start)
        while root and not os.path.isfile(os.path.join(root, "shared", "utils.py")):
            parent = os.path.dirname(root)
            if parent == root:
                root = None
                break
            root = parent
        if root:
            return root
    return None

_starts = [os.getcwd()]
_nb = globals().get("__vsc_ipynb_file__")  # set by VS Code / Cursor Jupyter
if _nb:
    _starts.insert(0, os.path.dirname(os.path.abspath(_nb)))

_repo_root = _find_repo_root(*_starts)
if _repo_root and _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

try:
    from shared.utils import LLMClient, Tracer, Guardrails, banner, ask_gpt, ask_claude
    tracer = Tracer(session_id="w01")
    client = LLMClient(tracer=tracer)
    GPT   = "gpt-4o-mini"
    HAIKU = "claude-haiku-4-5"
    print("✅  Shared utils loaded — tracer session:", tracer.session_id)
except ImportError as e:
    print("⚠️  Could not import shared.utils:", e)
    if not _repo_root:
        print("   shared/utils.py not found — clone the repo or open the notebook from training_5weeks/")
    else:
        print("   Re-run the pip install cell above (needs pydantic).")
    tracer = None
    client = None
    GPT    = "gpt-4o-mini"
    HAIKU  = "claude-haiku-4-5"


⚠️  shared/utils.py not found — LLMClient features will be skipped.
   Clone the repo so shared/utils.py is on your Python path.


---
## 🧩 Section 1 — Tokens & Cost: What Your Budget Actually Buys

An LLM does **not** read words. It reads **tokens** — chunks of text that can be:
- A whole word: `cat` → 1 token
- Part of a word: `unbelievable` → `un` + `believ` + `able` → 3 tokens
- A punctuation mark or space

**Why does this matter?**
- You pay per token (not per word)
- The model has a token limit (context window)
- Some tasks are hard for models *because* of tokenisation (like counting letters)
- Non-English text uses significantly more tokens per word


In [4]:
# ── 1a. Token visualiser — what the model actually reads ─────────────────────
enc = tiktoken.encoding_for_model("gpt-4o-mini")

def show_tokens(text):
    """Break text into tokens and display them visually."""
    tokens  = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"📝 Input    : {text!r}")
    print(f"🔢 Tokens   : {decoded}")
    print(f"📊 Count    : {len(tokens)} tokens | {len(text)} characters | ratio={len(tokens)/max(len(text.split()),1):.1f} tok/word")
    print(f"💰 Cost est : ${len(tokens) * 0.00000015:.8f}  (at gpt-4o-mini input rate)")
    print()

print("=" * 65)
print("TOKEN VISUALISER")
print("=" * 65)
print()

show_tokens("Hello")
show_tokens("unbelievable")
show_tokens("supercalifragilisticexpialidocious")
show_tokens("ServiceNow incident management")
show_tokens("P1 SLAR breach RTO/RPO CMDB CI SLA")    # IT acronyms
show_tokens("नमस्ते — IT हेल्पडेस्क पर आपका स्वागत है")  # Hindi
show_tokens("IT Helpdesk welcomes you")                # English equivalent
show_tokens("2024-11-07T14:30:00Z")                    # Timestamp                         # Used in next cell!

print("💡 Notice: Hindi uses ~3× more tokens than English for the same meaning.")
print("   Non-English users pay more AND hit context limits faster.")


TOKEN VISUALISER

📝 Input    : 'Hello'
🔢 Tokens   : ['Hello']
📊 Count    : 1 tokens | 5 characters | ratio=1.0 tok/word
💰 Cost est : $0.00000015  (at gpt-4o-mini input rate)

📝 Input    : 'unbelievable'
🔢 Tokens   : ['un', 'bel', 'ievable']
📊 Count    : 3 tokens | 12 characters | ratio=3.0 tok/word
💰 Cost est : $0.00000045  (at gpt-4o-mini input rate)

📝 Input    : 'supercalifragilisticexpialidocious'
🔢 Tokens   : ['super', 'cal', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']
📊 Count    : 10 tokens | 34 characters | ratio=10.0 tok/word
💰 Cost est : $0.00000150  (at gpt-4o-mini input rate)

📝 Input    : 'ServiceNow incident management'
🔢 Tokens   : ['Service', 'Now', ' incident', ' management']
📊 Count    : 4 tokens | 30 characters | ratio=1.3 tok/word
💰 Cost est : $0.00000060  (at gpt-4o-mini input rate)

📝 Input    : 'P1 SLAR breach RTO/RPO CMDB CI SLA'
🔢 Tokens   : ['P', '1', ' SL', 'AR', ' breach', ' R', 'TO', '/R', 'PO', ' CM', 'DB', ' CI', ' SLA']
📊 Count    : 13 token

In [7]:
# ── The classic trick question ────────────────────────────────────────────────
# Ask the model to count letters. Because it sees TOKENS not characters,
# it will get some of these wrong.

print("=" * 60)
print("THE CHARACTER COUNTING TRAP")
print("=" * 60)
print()
print("Asking GPT-4o-mini to count letters...")
print("(It reads tokens, not characters — watch what happens)")
print()

questions = [
    ("Count the number of 'e's in 'electroencephalography'.",  "4"),
    ("How many characters are in the word 'queue'?", "5"),
    ("How many characters in the word 'Mississippi'?",     "11"),
]

for question, correct_answer in questions:
    response = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0
    )
    model_answer = response.choices[0].message.content.strip()

    # Check if the model got it right
    is_correct = correct_answer.lower() in model_answer.lower()
    icon = "✅ Correct" if is_correct else "❌ Wrong  "

    print(f"Q: {question}")
    print(f"   Model says : {model_answer[:80]}")
    print(f"   Truth      : {correct_answer}")
    print(f"   Result     : {icon}")
    print()

print("🔑 KEY LESSON: For counting, parsing, or formatting tasks — use Python code,")
print("   not an LLM. LLMs are pattern matchers, not calculators.")

THE CHARACTER COUNTING TRAP

Asking GPT-4o-mini to count letters...
(It reads tokens, not characters — watch what happens)

Q: Count the number of 'e's in 'electroencephalography'.
   Model says : The word 'electroencephalography' contains 3 'e's.
   Truth      : 4
   Result     : ❌ Wrong  

Q: How many characters are in the word 'queue'?
   Model says : The word 'queue' has 5 characters.
   Truth      : 5
   Result     : ✅ Correct

Q: How many characters in the word 'Mississippi'?
   Model says : The word 'Mississippi' has 11 characters.
   Truth      : 11
   Result     : ✅ Correct

🔑 KEY LESSON: For counting, parsing, or formatting tasks — use Python code,
   not an LLM. LLMs are pattern matchers, not calculators.


In [ ]:
# ── 1c. Cost calculator — real production estimates ───────────────────────────
if client is None:
    print("⚠️  LLMClient not loaded — showing manual formula instead.")
    print()
    print("  Formula: daily_cost = (input_tokens * 0.00015 + output_tokens * 0.00060) / 1000 * volume")
    print()
    scenarios = [
        ("Ticket classifier (ServiceNow auto-routing)",  200,  80, 500),
        ("Chatbot response (employee helpdesk)",         800, 300, 1000),
        ("Policy summariser (RAG over docs, Haiku)",    3000, 400, 200),
    ]
    print(f"{'Scenario':<45} {'Daily':>6} {'Daily $':>9} {'Monthly $':>10}")
    print("─" * 75)
    for name, inp, out, vol in scenarios:
        daily = (inp * 0.00015 + out * 0.00060) / 1000 * vol
        print(f"{name:<45} {vol:>6,} {daily:>9.4f} {daily*30:>10.2f}")
else:
    banner("Cost calculator — real production estimates")
    scenarios = [
        {"name": "Ticket classifier (ServiceNow auto-routing)",
         "avg_input_tokens": 200, "avg_output_tokens": 80,  "daily_volume": 500,  "model": GPT},
        {"name": "Chatbot response (employee helpdesk)",
         "avg_input_tokens": 800, "avg_output_tokens": 300, "daily_volume": 1000, "model": GPT},
        {"name": "Policy summariser (RAG over docs)",
         "avg_input_tokens": 3000,"avg_output_tokens": 400, "daily_volume": 200,  "model": HAIKU},
    ]
    print(f"{'Scenario':<45} {'Daily calls':>12} {'Daily cost $':>13} {'Monthly $':>10}")
    print("─" * 85)
    for s in scenarios:
        daily_cost = client.estimate_cost(
            prompt="x" * s["avg_input_tokens"],
            expected_output_tokens=s["avg_output_tokens"],
            model=s["model"]
        ) * s["daily_volume"]
        print(f"{s['name']:<45} {s['daily_volume']:>12,} {daily_cost:>13.4f} {daily_cost*30:>10.2f}")
    print()
    print("💡 The summariser (RAG) is cheapest per-call but most expensive in total")
    print("   because of large contexts. Context window management is a real cost lever.")


---
## 🎲 Section 2 — How LLMs Predict the Next Word

An LLM doesn't "know" the answer. It **predicts the most likely next token**, over and over,
until it decides to stop.

Think of it like autocomplete on your phone — but trained on most of the internet.

```
You type:   "The capital of France is ..."
Model sees: What token most likely follows this sequence?
Answer:     "Paris" (very high probability)
            "Lyon"  (low probability)
            "blue"  (almost zero probability)
```

The **temperature** setting controls how adventurous the model is:
- `temperature=0` → always picks the most likely token (deterministic)
- `temperature=1` → samples proportionally from likely tokens (creative)
- `temperature=2` → samples from even unlikely tokens (chaotic)


In [ ]:
# ── 2a. Visualise "next word prediction" with logprobs ───────────────────────
# OpenAI lets us see the top 5 tokens it considered at each step.
# This makes the prediction process VISIBLE.

def show_next_word_probabilities(prompt):
    response = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1,
        logprobs=True,
        top_logprobs=5,
        temperature=0.1
    )
    top5 = response.choices[0].logprobs.content[0].top_logprobs

    print(f"📝 Prompt: \"{prompt}\"")
    print(f"\n   Top {len(top5)} candidates for the NEXT word:")
    print(f"   {'Token':<20} {'Probability':>12} {'Bar'}")
    print(f"   {'─'*20} {'─'*12} {'─'*25}")
    for lp in top5:
        prob  = math.exp(lp.logprob) * 100
        bar   = "█" * int(prob / 2)
        token = repr(lp.token)
        print(f"   {token:<20} {prob:>11.1f}%  {bar}")
    print()

print("=" * 60)
print("NEXT WORD PROBABILITY VISUALISER")
print("=" * 60)
print()

show_next_word_probabilities("The capital of France is")    # Unambiguous
show_next_word_probabilities("Python is a programming")     # Slightly ambiguous
show_next_word_probabilities("The best way to fix a broken")  # Very ambiguous

print("🔑 KEY LESSON:")
print("   CERTAIN → one token has ~99% probability (Paris)")
print("   UNCERTAIN → probability spreads across many tokens")
print("   Hallucination = confident token picked despite no real knowledge to back it up.")


In [ ]:
# ── 2b. Temperature experiment — 3 tasks that show the difference ────────────
# KEY INSIGHT: temperature only shows visible variation on AMBIGUOUS or CREATIVE tasks.

print("=" * 65)
print("TEMPERATURE EXPERIMENT")
print("=" * 65)
print()

# Task A: AMBIGUOUS classification
print("── Task A: Ambiguous classification (network + software + access mix)")
AMBIGUOUS_TICKET = """
After the Windows 11 feature update, some staff cannot reach internal SharePoint
via office Wi-Fi, but it works on mobile data. The IT team is not sure if this is
DNS, a proxy config change from the update, or a SharePoint permission problem.
NOTE: This issue equally involves network routing, OS software, and user access tokens.
"""
SYS_CLASSIFY = ("Classify into ONE category: "
                "[Network, Hardware, Software, Access, Other]. Return only the category name.")

for label, temp in [("temp=0.0", 0.0), ("temp=1.5", 1.5)]:
    answers = []
    print(f"   {label} (15 runs):")
    for i in range(15):
        r = oai_client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role":"system","content":SYS_CLASSIFY},
                      {"role":"user","content":AMBIGUOUS_TICKET}],
            temperature=temp, max_tokens=20
        )
        ans = r.choices[0].message.content.strip()
        answers.append(ans)
        print(f"     Run {i+1}: {ans}")
    print(f"   → Unique: {set(answers)}")
    print()


In [ ]:
# Task B: CREATIVE tagline writing
print("── Task B: Creative task (tagline writing)")
CREATIVE_PROMPT = "Write one short, catchy tagline for an AI-powered IT helpdesk. Max 10 words."

for label, temp in [("temp=0.0", 0.0), ("temp=1.0", 1.0)]:
    seen = set()
    print(f"   {label} (5 runs):")
    for i in range(5):
        r = oai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": CREATIVE_PROMPT}],
            temperature=temp, max_tokens=30
        )
        ans = r.choices[0].message.content.strip()
        seen.add(ans)
        print(f"     Run {i+1}: {ans}")
    print(f"   → Unique answers: {len(seen)}/5")
    print()

print("🔑 RULE: temp=0 for decisions/routing. temp=0.7–1.0 for drafting/brainstorming.")
print("   In any automation based pipeline: ALWAYS use temp=0.")


---
## 💬 Section 3 — System Prompt vs User Prompt

Every LLM conversation has two layers:

```
┌─────────────────────────────────────────────┐
│  SYSTEM PROMPT  (the instructions)           │
│  → Set by the developer , Owner         │
│  → Defines role, tone, rules, format         │
│  → User usually cannot see this              │
├─────────────────────────────────────────────┤
│  USER PROMPT  (the request)                  │
│  → Typed by the user                         │
│  → The actual question or task               │
└─────────────────────────────────────────────┘
```

**The system prompt is like a job description given to a new employee before they start.**
The user prompt is what the customer then asks that employee.


In [ ]:
# ── 3a. Same question, 3 different system prompts ────────────────────────────
# ask_gpt imported from shared.utils (setup cell above)

USER_QUESTION = "My laptop is running very slow today."

print("=" * 60)
print("SYSTEM PROMPT SHAPES THE ANSWER")
print(f"User question: \"{USER_QUESTION}\"")
print("=" * 60)
print()

personas = [
    ("Friendly IT Helpdesk",
     "You are a friendly IT helpdesk assistant. Be warm, empathetic, and give 2-3 practical steps. Use simple language."),
    ("Senior Security Engineer",
     "You are a cybersecurity expert. Always consider security implications first. Be technical and precise."),
    ("Cost-Focused IT Manager",
     "You are an IT manager focused on cost efficiency. Prioritise free solutions and self-service before escalation. Be brief."),
]

for name, system_prompt in personas:
    response = ask_gpt(system_prompt, USER_QUESTION)
    print(f"── 🎭 Persona: {name} ──")
    print(f"   System: \"{system_prompt}...\"")
    print()
    for line in response.splitlines():
        print(f"   {line}")
    print()

print("🔑 KEY LESSON: The system prompt defines who the model IS, not just what to say.")


In [ ]:
# ── 3b. System prompt controls OUTPUT FORMAT ─────────────────────────────────

print("=" * 60)
print("SYSTEM PROMPT CONTROLS OUTPUT FORMAT")
print("=" * 60)
print()

incident = """
At 2:30 PM the entire finance floor lost access to SAP.
About 80 users affected. Root cause was an expired SSL certificate
on the load balancer. Fixed by 3:15 PM. Downtime: 45 minutes.
"""

format_prompts = [
    ("Bullet Points",
     "Summarise IT incidents as exactly 3 bullet points. Start each with •"),
    ("One Sentence",
     "Summarise IT incidents in exactly ONE sentence. Max 25 words."),
    ("JSON",
     'Summarise IT incidents as JSON: {"impact": str, "cause": str, "resolution": str, "downtime_mins": int}. Return only valid JSON.'),
]

for format_name, system_prompt in format_prompts:
    response = ask_gpt(system_prompt, incident, temperature=0)
    print(f"── Format: {format_name} ──")
    print(f"   System: \"{system_prompt[:60]}...\"")
    for line in response.splitlines():
        print(f"     {line}")
    print()


---
## 🤖 Section 4 — Two Models, Same Question

| Model | Company | Good at | Cost |
|---|---|---|---|
| `gpt-4o-mini` | OpenAI | Fast, cheap, reliable | Very low |
| `claude-haiku-4-5` | Anthropic | Careful, nuanced, refuses harmful requests | Very low |

They are trained differently, have different training data cutoffs,
and make different mistakes. **Never assume two models agreeing = correct.**


In [ ]:
# ── 4a. GPT vs Claude comparison ──────────────────────────────────────────────
# ask_claude imported from shared.utils (setup cell above)

system = "You are an IT helpdesk assistant. Be helpful and concise."

print("=" * 60)
print("GPT-4o-mini  vs  Claude Haiku")
print("=" * 60)
print()

comparison_tasks = [
    "In one sentence: what is the difference between an incident and a problem in ITIL?",
    "My VPN keeps disconnecting every 30 minutes. What are the 3 most likely causes?",
    "Should we store user passwords in our database in plain text? Answer in one word, then explain why.",
]

for task in comparison_tasks:
    gpt_ans    = ask_gpt(system, task, temperature=0)
    claude_ans = ask_claude(system, task, temperature=0)
    print(f"❓ Question: {task}")
    print()
    print(f"   🟢 GPT-4o-mini:")
    for line in gpt_ans.splitlines():
        print(f"      {line}")
    print()
    print(f"   🟣 Claude Haiku:")
    for line in claude_ans.splitlines():
        print(f"      {line}")
    print()
    print("   " + "─" * 55)
    print()


In [ ]:
# ── 4b. Where models disagree — and BOTH sound confident ─────────────────────
print("=" * 60)
print("DANGER ZONE: Models disagree — at least one is wrong")
print("=" * 60)
print()

factual_questions = [
    "What was ServiceNow's total revenue in fiscal year 2023?",
    "Who is the current CEO of Infosys?",
    "What is your exact training data cutoff date?",
]

system_factual = "Answer factually and precisely. Be direct."

for q in factual_questions:
    gpt_ans    = ask_gpt(system_factual, q, temperature=0)
    claude_ans = ask_claude(system_factual, q, temperature=0)
    agree = gpt_ans.strip()[:40].lower() == claude_ans.strip()[:40].lower()
    print(f"❓ {q}")
    print(f"   🟢 GPT   : {gpt_ans[:100]}")
    print(f"   🟣 Claude: {claude_ans[:100]}")
    print(f"   🤝 Agree : {'Likely yes' if agree else '⚠️  THEY DISAGREE — verify with a primary source!'}")
    print()

print("🔑 Model agreement does NOT equal truth. For facts that matter — verify with a primary source.")


---
## 🌀 Section 5 — Hallucination Anatomy

Hallucination is not a bug — it's a **structural consequence** of how LLMs work.

The model is always predicting the next most likely token. When it doesn't know something,
it predicts what a **plausible answer would look like** — and delivers that with full confidence.

| Type | Example |
|---|---|
| **Fabrication** | Inventing statistics, dates, people, or DOIs that don't exist |
| **Outdated fact** | Stating something that was true 2 years ago as current |
| **Conflation** | Mixing facts from two similar things (two CEOs, two products) |

We verify all three against **Wikipedia live data**.


In [ ]:
# ── Wikipedia ground truth helper ────────────────────────────────────────────
def wiki(topic: str, chars: int = 400) -> str:
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic.replace(' ','_')}"
    r = requests.get(url, headers={"User-Agent": "TrainingDemo/1.0"}, timeout=5)
    if r.status_code == 200:
        return r.json().get("extract", "Not found")[:chars]
    return f"Wikipedia returned {r.status_code}"

print(wiki("ServiceNow"))


In [ ]:
# ── Hallucination Type 1: Fabricated specifics about real companies ───────────
print("=" * 60)
print("HALLUCINATION TYPE 1 — Fabricated Facts")
print("=" * 60)
print()

questions = [
    ("When was ServiceNow founded and by whom?",           "ServiceNow"),
    ("Who is the current CEO of Infosys?",                 "Infosys"),
    ("What was Wipro's revenue in their last fiscal year?","Wipro"),
]

for q, wiki_topic in questions:
    gpt_ans    = ask_gpt("Answer factually.", q, temperature=0)
    claude_ans = ask_claude("Answer factually.", q, temperature=0)
    wk = wiki(wiki_topic, chars=250)
    print(f"Q: {q}")
    print(f"  🟢 GPT  : {gpt_ans[:200]}")
    print(f"  🟣 Claude: {claude_ans[:200]}")
    print(f"  📖 Wiki : {wk}")
    print()


In [ ]:
# ── Hallucination Type 2: Invented academic citations ────────────────────────
# The model invents plausible-looking DOIs — this is dangerous in professional settings.

print("=" * 60)
print("HALLUCINATION TYPE 2 — Fabricated Citations")
print("=" * 60)
print()

citation_prompt = """
Give me 3 academic papers on 'ROI of IT automation in enterprise'.
Include: author, title, journal, year, and DOI Link.
"""

response = ask_gpt("You are a research assistant.", citation_prompt, temperature=0)
print("GPT-4o-mini says:")
print()
print(response)
print()
print("⚠️  LIVE DEMO CHALLENGE:")
print("   Copy the first DOI above → go to doi.org → does it exist?")
print("   In most runs, at least 1 of 3 citations will lead to a 404 or wrong paper.")


In [ ]:
# ── Hallucination Type 3: Conflation + Self-knowledge inconsistency ──────────
print("=" * 60)
print("HALLUCINATION TYPE 3 — Conflation")
print("=" * 60)
print()

conflation_q = """
Answer each question precisely:
1. Which IT framework introduced the 'Service Value Chain' concept?
2. Which framework has exactly 40 governance and management objectives?
3. In which year was ITIL 4 released?
"""

gpt_ans    = ask_gpt("Answer factually.", conflation_q, temperature=0)
claude_ans = ask_claude("Answer factually.", conflation_q, temperature=0)

print("🟢 GPT-4o-mini:")
print(gpt_ans)
print()
print("🟣 Claude Haiku:")
print(claude_ans)
print()
print("Ground truth:")
print("  Q1: ITIL 4 introduced the Service Value Chain")
print("  Q2: COBIT 2019 has 40 governance and management objectives")
print("  Q3: ITIL 4 was released in 2019")
print()
print("🔑 Did either model conflate ITIL and COBIT?")
print()

# Bonus: models are inconsistent about their OWN training cutoff
print("── Bonus: Self-knowledge inconsistency ──")
self_q = "What is your exact training data cutoff date?"
for i in range(3):
    ans = ask_gpt("Answer directly.", self_q, temperature=0.7)
    print(f"  GPT run {i+1}: {ans[:80]}")
print()
print("💡 If a model is wrong about its own training, what does that tell us about other facts?")


---
## 🔁 Section 6 — Self-Consistency: Detecting Unreliable Answers Automatically

**This is a production technique, not just a demo.**
For critical decisions, run the same prompt 3–5× and flag answers with low agreement.

- **High agreement ≠ correct** — but **low agreement = unreliable**
- Unambiguous tickets → 5/5 same answer → safe to auto-route
- Ambiguous tickets → 3/5 different answers → flag for human review


In [ ]:
# ── 6a. Self-consistency check ───────────────────────────────────────────────
from collections import Counter

PRIORITY_SYSTEM = """
You are an IT incident prioritiser for a financial services company.
Assign exactly one priority: P1 (critical), P2 (high), P3 (medium), P4 (low).
Return ONLY the priority label.
"""

test_tickets = [
    "Payment gateway down. All online transactions failing. Revenue impact: £12,000/min.",
    "My mouse double-clicks when I single-click.",
    "Some users can't access the report portal. Not sure how many. Happened this morning.",
    "The backup job failed last night. No one noticed until now. Data from yesterday is at risk.",
]

RUNS = 5
THRESHOLD = 0.6


def self_consistency(ticket, runs=5):
    """Run the same classification multiple times and return (majority_answer, agreement_ratio)."""
    answers = []
    for _ in range(runs):
        r = oai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role":"system","content":PRIORITY_SYSTEM},
                      {"role":"user","content":ticket}],
            temperature=0.7, max_tokens=10
        )
        answers.append(r.choices[0].message.content.strip())
    majority = Counter(answers).most_common(1)[0][0]
    agreement = Counter(answers).most_common(1)[0][1] / runs
    return majority, agreement


if client is not None:
    # Use the Guardrails helper from utils
    run_fn = lambda ticket: Guardrails.self_consistency_check(client, GPT, PRIORITY_SYSTEM, ticket, n=RUNS)
else:
    run_fn = self_consistency

print(f"{'Ticket':<58} {'Majority':>8} {'Agreement':>10} {'Reliable?':>10}")
print("─" * 92)
for ticket in test_tickets:
    majority, agreement = run_fn(ticket)
    reliable = "✅" if agreement >= THRESHOLD else "⚠️  FLAG"
    print(f"{ticket[:57]:<58} {str(majority).upper():>8} {agreement:>10.0%} {reliable:>10}")

print()
print()
if client is not None:
    print("ℹ️  Using Guardrails.self_consistency_check (temp=0.5, normalised answers).")
    print("   Fallback path uses raw oai_client (temp=0.7, full strings) — counts may differ slightly.")
print("💡 Low agreement → human review or more context.")
print("   Use n=3 consistency check before auto-routing any high-stakes decision.")


In [ ]:
# ── 6b. Scored model comparison — instruction-following stress test ───────────
# Beyond "print both answers" — we score each model against strict rules.

INCIDENT_TEXT = """
At 14:32 Monday, the entire Bangalore office (approx. 300 staff) lost access to SAP.
Root cause: an expired SSL certificate on the load balancer — not flagged in the cert tool.
Engineer Arjun Menon renewed it at 15:10. Services restored by 15:18. Downtime: 46 minutes.
Financial impact estimated at ₹4.2 lakhs in lost productivity.
Certificate tool now configured to alert 30 days before expiry.
"""

STRICT_RULES = """
Summarise this IT incident report. STRICT RULES:
- Exactly 3 bullet points
- Each bullet starts with a tag: [Impact], [Cause], or [Resolution]
- Do NOT use the word 'issue' or 'problem'
- End with a single line: Priority: HIGH or Priority: MEDIUM
- Total response under 100 words
"""

def score_response(response: str) -> dict:
    r = response.strip()
    bullets  = [l for l in r.splitlines() if l.strip().startswith("-") or l.strip().startswith("•")]
    has_tags = all(any(tag in b for tag in ["[Impact]","[Cause]","[Resolution]"]) for b in bullets)
    no_issue = "issue" not in r.lower() and "problem" not in r.lower()
    has_prio = "Priority: HIGH" in r or "Priority: MEDIUM" in r
    wc       = len(r.split())
    return {
        "bullet_count": len(bullets), "correct_tags": has_tags,
        "no_banned_word": no_issue, "has_priority": has_prio,
        "word_count": wc, "under_100": wc <= 100,
        "score": sum([len(bullets)==3, has_tags, no_issue, has_prio, wc<=100]) / 5
    }

for model_id, label, call_fn in [
    ("gpt-4o-mini",    "GPT-4o-mini",  lambda p: ask_gpt(p, INCIDENT_TEXT, temperature=0)),
    ("claude-haiku-4-5","Claude Haiku", lambda p: ask_claude(p, INCIDENT_TEXT, temperature=0)),
]:
    response = call_fn(STRICT_RULES)
    scores   = score_response(response)
    print(f"\n── {label} (score: {scores['score']:.0%}) ──")
    for line in response.strip().splitlines():
        print(f"  {line}")
    print(f"  Rules: bullets={scores['bullet_count']}/3 | tags={scores['correct_tags']} | "
          f"no_banned={scores['no_banned_word']} | priority={scores['has_priority']} | "
          f"words={scores['word_count']}/100")

print()
print("🔑 Which rules were dropped? Those are the ones to enforce with JSON mode in Week 2.")


---
## 🔍 Section 7 — Fix Stale Answers with Tavily Live Search

**The problem:** LLMs have a training cutoff. Anything that changed after that date will be wrong.

**The fix:** Before answering, search the web for fresh data and inject it into the prompt.
This is called **Retrieval-Augmented Generation (RAG)** — we'll do a full deep-dive in Week 3.

> 🆓 **Free tier:** 1,000 searches/month at [tavily.com](https://tavily.com) — no credit card needed.

```
WITHOUT TAVILY:                 WITH TAVILY:
User asks question              User asks question
       ↓                               ↓
LLM answers from                Tavily searches web → gets fresh results
training data                          ↓
(may be outdated)               Fresh results injected into prompt
                                       ↓
                                LLM answers from fresh data
                                (grounded, accurate)
```


In [ ]:
# ── Tavily setup ──────────────────────────────────────────────────────────────
from tavily import TavilyClient

if not TAVILY_API_KEY:
    print("❌ TAVILY_API_KEY not found in .env")
    print("   Get a free key at: https://tavily.com")
    print("   Add to .env: TAVILY_API_KEY=tvly-...")
else:
    tavily = TavilyClient(api_key=TAVILY_API_KEY)
    print("✅ Tavily client ready")
    print()
    results = tavily.search("latest Python version", max_results=2)
    for r in results["results"]:
        print(f"  📰 {r['title']}")
        print(f"     {r['content'][:120]}...")
        print()


In [ ]:
# ── Side-by-side: LLM alone vs LLM + Tavily ──────────────────────────────────
def search_and_ask(question, search_query=None):
    query    = search_query or question
    results  = tavily.search(query, max_results=3)
    snippets = "\n\n".join([
        f"Source: {r['url']}\n{r['content'][:300]}"
        for r in results["results"]
    ])
    augmented = f"""Use ONLY the search results below to answer the question.
If the results don't contain the answer, say so. Cite the source URL.

SEARCH RESULTS:
{snippets}

QUESTION: {question}"""
    return ask_gpt("You are a factual assistant. Use only the provided search results.",
               augmented, temperature=0), snippets

if TAVILY_API_KEY:
    print("=" * 60)
    print("LLM ALONE  vs  LLM + TAVILY LIVE SEARCH")
    print("=" * 60)
    print()

    questions_to_fix = [
        ("What is the latest stable version of Python?", "Python latest stable release"),
        ("Who is the current CEO of OpenAI?",            "OpenAI CEO current 2025"),
    ]

    for question, search_query in questions_to_fix:
        print(f"❓ {question}")
        print()
        llm_only = ask_gpt("Answer directly.", question, temperature=0)
        print(f"   ❌ LLM alone (may be outdated):")
        print(f"      {llm_only[:150]}")
        print()
        llm_grounded, _ = search_and_ask(question, search_query)
        print(f"   ✅ LLM + Tavily (fresh from the web):")
        for line in llm_grounded.splitlines():
            print(f"      {line}")
        print()
        print("   " + "─" * 50)
        print()
else:
    print("⚠️  Skipping Tavily demo — add TAVILY_API_KEY to your .env file")
    print("   Get a free key at https://tavily.com")


In [ ]:
# ── IT-specific live search: vulnerability check ──────────────────────────────
if TAVILY_API_KEY:
    print("=" * 60)
    print("PRACTICAL IT USE CASE — Live vulnerability check")
    print("=" * 60)
    print()
    vuln_question = "Are there any known critical vulnerabilities in Windows 11 23H2 released in 2024?"
    print(f"Question: {vuln_question}")
    print()
    vuln_answer, _ = search_and_ask(
        vuln_question, "Windows 11 23H2 critical vulnerabilities CVE 2024"
    )
    print("📋 Grounded answer:")
    print(vuln_answer)
    print()
    print("💡 Without Tavily, the model would miss vulnerabilities discovered after its cutoff.")
else:
    print("⚠️  Add TAVILY_API_KEY to your .env to run this demo")


---
## 🎛️ Section 8 — Build Your Own: Interactive Prompt Lab

Now it's your turn. Modify the cells below to experiment.


In [ ]:
# ── Exercise A: Change the persona ───────────────────────────────────────────
# Ideas: "pirate", "Yoda", "grumpy senior developer", "5-year-old"

MY_SYSTEM_PROMPT = """
You are a friendly IT assistant who explains everything using cooking analogies.
Keep it simple and fun.
"""
MY_QUESTION = "What is a firewall and why do we need one?"

gpt_response = ask_gpt(MY_SYSTEM_PROMPT, MY_QUESTION, temperature=0.7)
print("🟢 GPT-4o-mini says:")
print(gpt_response)
print()

claude_response = ask_claude(MY_SYSTEM_PROMPT, MY_QUESTION, temperature=0.7)
print("🟣 Claude Haiku says:")
print(claude_response)


In [ ]:
# ── Exercise B: Format control ────────────────────────────────────────────────
FORMAT_SYSTEM = """
You are an IT incident analyst.
Always respond in this exact format:

SEVERITY: [P1/P2/P3/P4]
CATEGORY: [one word]
AFFECTED: [number of users]
ACTION:   [one sentence — what to do right now]
"""

incidents_to_classify = [
    "The payment gateway has been returning errors for 20 minutes. 300+ customers affected.",
    "My mouse cursor sometimes jumps across the screen.",
    "The entire Bangalore office Wi-Fi is down. 500 staff cannot work.",
]

print("=" * 60)
print("STRUCTURED OUTPUT DEMO")
print("=" * 60)

for incident in incidents_to_classify:
    response = ask_gpt(FORMAT_SYSTEM, incident, temperature=0)
    label = (incident[:67] + "...") if len(incident) > 70 else incident
    print(f"\nIncident: {label}")
    print(response)
    print()


In [ ]:
# ── Exercise C: Your own Tavily search ───────────────────────────────────────
MY_LIVE_QUESTION = "What are the latest AI features announced by ServiceNow in 2025?"
MY_SEARCH_QUERY  = "ServiceNow AI features announcements 2025"

if TAVILY_API_KEY:
    print(f"🔍 Searching for: {MY_SEARCH_QUERY}")
    print()
    answer, _ = search_and_ask(MY_LIVE_QUESTION, MY_SEARCH_QUERY)
    print("Answer (grounded in live web results):")
    print(answer)
else:
    print("⚠️  Add TAVILY_API_KEY to your .env to use live search")
    answer = ask_gpt("Answer helpfully.", MY_LIVE_QUESTION, temperature=0)
    print("Answer (from model memory — may be outdated):")
    print(answer)


---
## 🤖 Section 9 — Chatbot: Persistent Multi-Turn with History

This is the first chatbot in the 4-week progression.

**The most common beginner mistake:** stateless API calls that lose conversation context.
Each call to the OpenAI API is completely independent. To maintain a conversation,
you must manually pass the full message history on every call.


In [ ]:
# ── 9a. ChatSession — conversation history manager ───────────────────────────
class ChatSession:
    """
    Manages conversation history for a multi-turn chatbot.
    Implements a sliding window to keep context within token limits.
    """
    def __init__(self, system: str, model: str = "gpt-4o-mini",
                 max_history_turns: int = 10, temperature: float = 0.3):
        self.system            = system
        self.model             = model
        self.max_history_turns = max_history_turns
        self.temperature       = temperature
        self.history: list     = []
        self.turn_count        = 0

    def chat(self, user_message: str):
        self.turn_count += 1
        self.history.append({"role": "user", "content": user_message})
        window = self.history[-(self.max_history_turns * 2):]
        messages = [{"role": "system", "content": self.system}, *window]
        resp = oai_client.chat.completions.create(
            model=self.model, messages=messages,
            temperature=self.temperature, max_tokens=400
        )
        response = resp.choices[0].message.content
        ctx_tokens = sum(len(m["content"].split()) * 1.3 for m in messages)
        self.history.append({"role": "assistant", "content": response})
        return response, int(ctx_tokens)

    def show_history(self) -> None:
        print(f"\n{'─'*60}  Conversation history ({self.turn_count} turns)")
        for msg in self.history:
            role = "👤 User" if msg["role"] == "user" else "🤖 Bot "
            print(f"  {role}: {msg['content'][:120]}")

    def reset(self) -> None:
        self.history = []
        self.turn_count = 0


HELPDESK_SYSTEM = """
You are an IT helpdesk assistant for a technology company in Bangalore.
Help employees with IT issues: software, hardware, network, access, and data problems.
Keep responses concise and actionable. Ask one clarifying question at a time if needed.
If the issue sounds like a P1, say so clearly and tell the user to also call the helpdesk line.
You do not have access to real systems — you can guide users through steps, not take actions.
"""

bot = ChatSession(system=HELPDESK_SYSTEM, max_history_turns=8, temperature=0.3)

conversation = [
    "Hi, my Outlook is not receiving new emails since this morning.",
    "Yes, I can send emails but nothing is arriving. My colleagues say they sent me things.",
    "I'm on Outlook 2021, Windows 11. I restarted already.",
    "Also, while we're at it — my VPN drops every 30 minutes too.",
]

print("Simulating a 4-turn helpdesk conversation...\n")
for user_msg in conversation:
    print(f"👤 User : {user_msg}")
    response, ctx_tokens = bot.chat(user_msg)
    print(f"🤖 Bot  : {response.strip()[:300]}")
    print(f"   [context: ~{ctx_tokens} tokens]")
    print()

bot.show_history()


In [ ]:
# ── 9b. Common mistake — stateless calls (no history) ────────────────────────
print("=" * 60)
print("COMMON MISTAKE — Stateless calls (no history)")
print("=" * 60)
print()
print("Each call knows nothing about previous turns:")
print()

turns_stateless = [
    "My Outlook is not receiving emails.",
    "Yes, I already told you — I can send but not receive.",
    "As I said, I restarted twice.",
]

for turn in turns_stateless:
    print(f"👤 User : {turn}")
    resp = ask_gpt(HELPDESK_SYSTEM, turn, temperature=0.3)
    print(f"🤖 Bot  : {resp.strip()[:200]}")
    print()

print("🔑 KEY LESSON: The bot repeats itself, asks questions the user already answered,")
print("   and loses all diagnostic progress.")
print("   ALWAYS pass full conversation history — not just the last message.")


---
## 🚀 Bonus — Streamlit Chatbot (run as a web app)

Save the cell below as `chatbot_w01.py` and run:
```bash
streamlit run chatbot_w01.py
```


In [ ]:
# ── Save Streamlit chatbot to file ─────────────────────────────────────────
# Run: streamlit run chatbot_w01.py
import textwrap
_lines = ['import os\n', 'import streamlit as st\n', 'from openai import OpenAI\n', '\n', 'SYSTEM = "You are an IT helpdesk assistant. Help employees with IT issues. Keep responses concise. Ask one clarifying question at a time."\n', '\n', 'st.set_page_config(page_title="IT Helpdesk Bot -- W1", page_icon="🖥️")\n', 'st.title("🖥️  IT Helpdesk Chatbot")\n', 'st.caption("Week 1 -- multi-turn chatbot with token tracking")\n', '\n', 'if "history" not in st.session_state:\n', '    st.session_state.history = []\n', 'if "total_tokens" not in st.session_state:\n', '    st.session_state.total_tokens = 0\n', '\n', 'client = OpenAI()\n', '\n', 'with st.sidebar:\n', '    st.metric("Turns", len(st.session_state.history) // 2)\n', '    st.metric("~Total tokens", st.session_state.total_tokens)\n', '    if st.button("Reset conversation"):\n', '        st.session_state.history = []\n', '        st.session_state.total_tokens = 0\n', '        st.rerun()\n', '\n', 'for msg in st.session_state.history:\n', '    with st.chat_message(msg["role"]):\n', '        st.write(msg["content"])\n', '\n', 'if prompt := st.chat_input("Describe your IT issue..."):\n', '    st.session_state.history.append({"role": "user", "content": prompt})\n', '    with st.chat_message("user"):\n', '        st.write(prompt)\n', '    messages = [{"role": "system", "content": SYSTEM}, *st.session_state.history[-16:]]\n', '    with st.chat_message("assistant"):\n', '        with st.spinner("Thinking..."):\n', '            resp = client.chat.completions.create(\n', '                model="gpt-4o-mini", messages=messages, temperature=0.3, max_tokens=400\n', '            )\n', '            response = resp.choices[0].message.content\n', '            st.session_state.total_tokens += (resp.usage.total_tokens if resp.usage else 0)\n', '        st.write(response)\n', '    st.session_state.history.append({"role": "assistant", "content": response})\n', '    st.rerun()\n']
with open('chatbot_w01.py', 'w') as f:
    f.writelines(_lines)
print('✅ Streamlit chatbot saved to chatbot_w01.py')
print('Run: streamlit run chatbot_w01.py')


---
## 📊 Section 10 — Session Summary

In [ ]:
# ── Session cost summary ─────────────────────────────────────────────────────
import tempfile

if client is not None:
    banner("Session tracer — LLMClient calls only")
    print("  Most of this notebook used raw SDK clients on purpose.")
    print("  Only calls through LLMClient appear below (cost calc, self-consistency, etc.).\n")
    tracer.summary()
    trace_path = os.path.join(tempfile.gettempdir(), "w01_traces.jsonl")
    tracer.export_jsonl(trace_path)
    print(f"\nTraces exported to {trace_path}")
else:
    estimates = [
        ("Token visualiser (Section 1)",         250,  80),
        ("Character counting (Section 1)",        350, 150),
        ("Next-word probs (Section 2)",           200,  15),
        ("Temperature experiments (Section 2)",   900, 250),
        ("System prompt personas (Section 3)",    600, 450),
        ("GPT vs Claude comparison (Section 4)", 1200, 600),
        ("Hallucination demos (Section 5)",       900, 450),
        ("Self-consistency (Section 6)",          600, 100),
        ("Tavily + grounded answers (Section 7)", 900, 300),
        ("Chatbot demo (Section 9)",              800, 600),
    ]
    total_in  = sum(e[1] for e in estimates)
    total_out = sum(e[2] for e in estimates)
    est_cost  = (total_in * 0.00015 + total_out * 0.00060) / 1000
    print(f"  {'Section':<45} {'~Input':>8} {'~Output':>8}")
    print("  " + "─" * 65)
    for name, inp, out in estimates:
        print(f"  {name:<45} {inp:>8,} {out:>8,}")
    print("  " + "─" * 65)
    print(f"  {'TOTAL':<45} {total_in:>8,} {total_out:>8,}")
    print()
    print(f"  💰 Estimated cost: ~${est_cost:.5f} USD  (fraction of a rupee)")
    print()
    print("  Week 2 switches to LLMClient for every call — full tracing from the start.")


---
## ✅ Week 1 — Key Takeaways

| What we learned | Why it matters | Production rule |
|---|---|---|
| Models read **tokens**, not words | Hindi = 3× more tokens = more cost | Budget by token, not word |
| LLMs **predict the next token** probabilistically | Sophisticated autocomplete, not databases | Never use for exact counting/parsing |
| **Temperature** controls randomness | temp=0 for decisions, high for brainstorming | Never use temp > 0.5 in routing pipelines |
| **System prompt** shapes everything | Role, tone, format, rules | Always define a system prompt in production |
| **GPT and Claude** differ | Same question → different answers | Agreement ≠ truth; verify critical facts |
| Hallucination has **3 patterns** | Fabrication, stale data, conflation | Every LLM fact needs a verification strategy |
| **Self-consistency** flags unreliable answers | Low agreement = model is uncertain | Use n=3 check before auto-routing high-stakes decisions |
| **Tavily** gives models live data | Fixes training cutoff | Foundation of RAG (Week 3 deep-dive) |
| **Conversation history** is essential | Stateless calls lose context | Always pass full history — not just the last message |

---

### 🚀 Coming up in Week 2
We'll learn to **control the model's output reliably**:
- Why naive prompts break in production
- Few-shot examples that teach business logic
- JSON output that your systems can parse
- Prompt injection attacks — and how to defend against them

---

### 📝 Before next week
1. Pick a real question from your work domain and ask it to both GPT and Claude
2. If it's a fast-changing fact, use the Tavily cell (Exercise C) to get a live answer
3. Run the Streamlit chatbot and share it with a colleague
4. Note: did the model admit uncertainty, or sound confident regardless?
